# Практическая работа: Логистическая регрессия

Цели работы:

- познакомиться с моделью логистической регрессии;
- научиться обучать модель на задачах классификации;
- освоить базовые метрики качества (accuracy, confusion matrix, precision, recall);
- научиться интерпретировать коэффициенты модели.

In [1]:
# 0. Импорт библиотек

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

np.random.seed(0)

## 0. Генерация синтетических датасетов

Мы будем использовать:

- стандартный датасет Iris из библиотеки scikit-learn;
- синтетический датасет для кредитного скоринга;
- синтетический датасет для прогнозирования болезни растений.

Это делает практику полностью воспроизводимой, без внешних файлов.

In [2]:
def make_credit_data(n_samples=1000, random_state=42):
    rng = np.random.default_rng(random_state)

    # Возраст: 21–70
    age = rng.normal(loc=40, scale=10, size=n_samples).clip(21, 70)

    # Доход (тыс. руб.): 20–300
    income = rng.lognormal(mean=np.log(60), sigma=0.5, size=n_samples)
    income = income.clip(20, 300)

    # Стаж работы (лет)
    years_employed = rng.uniform(0, 30, size=n_samples)
    years_employed = np.minimum(years_employed, age - 18)
    years_employed = np.clip(years_employed, 0, None)

    # Количество просрочек
    past_due = rng.poisson(lam=1.0, size=n_samples).clip(0, 10)

    # Сумма кредита (тыс. руб.)
    credit_amount = income * rng.uniform(2, 8, size=n_samples) + rng.normal(0, 20, size=n_samples)
    credit_amount = credit_amount.clip(50, 2000)

    # Наличие залога
    collateral = rng.binomial(1, p=0.4, size=n_samples)

    # Логистическая модель вероятности дефолта
    beta0 = -3.0
    beta_age = -0.01
    beta_income = -0.005
    beta_years = -0.03
    beta_past_due = 0.6
    beta_amount = 0.001
    beta_collateral = -0.8

    z = (
        beta0
        + beta_age * age
        + beta_income * (income / 10)
        + beta_years * years_employed
        + beta_past_due * past_due
        + beta_amount * (credit_amount / 100)
        + beta_collateral * collateral
    )

    p_default = 1 / (1 + np.exp(-z))
    default = rng.binomial(1, p_default)

    data = pd.DataFrame({
        "age": age,
        "income": income,
        "years_employed": years_employed,
        "past_due": past_due,
        "credit_amount": credit_amount,
        "collateral": collateral,
        "default": default,
        "p_default_true": p_default,
    })

    return data


def make_plants_data(n_samples=1000, random_state=123):
    rng = np.random.default_rng(random_state)

    # Температура (°C)
    temp = rng.normal(loc=22, scale=5, size=n_samples).clip(10, 35)

    # Влажность (%)
    humidity = rng.beta(a=2, b=2, size=n_samples) * 70 + 30  # 30–100

    # Осадки (мм)
    rain = rng.exponential(scale=50, size=n_samples).clip(0, 300)

    # Плотность посадки
    density = rng.uniform(2, 10, size=n_samples)

    # Сорт (устойчивость): 0, 1, 2
    variety = rng.integers(0, 3, size=n_samples)

    # Логистическая модель вероятности болезни
    beta0 = -8.0
    beta_temp = 0.15
    beta_temp2 = -0.003
    beta_humidity = 0.05
    beta_rain = 0.01
    beta_density = 0.25
    beta_variety = -0.8

    z = (
        beta0
        + beta_temp * temp
        + beta_temp2 * temp**2
        + beta_humidity * (humidity / 10)
        + beta_rain * (rain / 50)
        + beta_density * density
        + beta_variety * variety
    )

    p_disease = 1 / (1 + np.exp(-z))
    disease = rng.binomial(1, p_disease)

    data = pd.DataFrame({
        "temp": temp,
        "humidity": humidity,
        "rain": rain,
        "density": density,
        "variety": variety,
        "disease": disease,
        "p_disease_true": p_disease,
    })

    return data


# Создаём датасеты
credit = make_credit_data(n_samples=1000, random_state=42)
plants = make_plants_data(n_samples=1000, random_state=123)

credit.head(), plants.head()

(         age     income  years_employed  past_due  credit_amount  collateral  \
 0  43.047171  58.247620        1.270383         2     307.459869           0   
 1  29.600159  41.666652       11.600159         1     195.613639           1   
 2  47.504512  48.769642       16.409634         1     301.677391           0   
 3  49.405647  82.376463        4.402297         2     251.835197           1   
 4  21.000000  60.089866        3.000000         1     368.843862           1   
 
    default  p_default_true  
 0        0        0.091568  
 1        0        0.020575  
 2        0        0.032650  
 3        0        0.036799  
 4        0        0.028572  ,
         temp   humidity        rain   density  variety  disease  \
 0  17.054393  75.575005  106.718507  9.878049        2        0   
 1  20.161067  81.502189  222.429218  7.006361        1        0   
 2  28.439626  74.449373   41.551512  3.727385        1        0   
 3  22.969872  50.533393   49.501932  4.143898        1    

## 1. Логистическая регрессия на Iris

Начнём с классического датасета Iris (ирисы Фишера).
Задача: по размерам цветка предсказать его вид.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names_iris = iris.feature_names
target_names_iris = iris.target_names

print("Размер X:", X_iris.shape)
print("Размер y:", y_iris.shape)
print("Признаки:", feature_names_iris)
print("Классы:", target_names_iris)

df_iris = pd.DataFrame(X_iris, columns=feature_names_iris)
df_iris["target"] = y_iris
df_iris.head()

Размер X: (150, 4)
Размер y: (150,)
Признаки: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Классы: ['setosa' 'versicolor' 'virginica']


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


**Задание 1.1.**  
1. Посмотрите описание признаков и классов.  
2. Как вы думаете, какие признаки лучше всего различают виды?

### 1.1 Бинарная классификация: setosa vs не‑setosa

In [4]:
# Преобразуем задачу в бинарную: setosa (1) vs остальные (0)
y_binary = (y_iris == 0).astype(int)

# Возьмём два признака для наглядности
X_two = df_iris[["sepal length (cm)", "petal length (cm)"]].values

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_two, y_binary, test_size=0.3, random_state=42, stratify=y_binary
)

X_train_b.shape, X_test_b.shape

((105, 2), (45, 2))

#### Обучение логистической регрессии (бинарный случай)

In [5]:
log_reg_iris_bin = LogisticRegression(solver="liblinear")

log_reg_iris_bin.fit(X_train_b, y_train_b)

y_pred_b = log_reg_iris_bin.predict(X_test_b)

print("Accuracy:", accuracy_score(y_test_b, y_pred_b))
print("Confusion matrix:\n", confusion_matrix(y_test_b, y_pred_b))
print("Classification report:\n", classification_report(y_test_b, y_pred_b))

Accuracy: 1.0
Confusion matrix:
 [[30  0]
 [ 0 15]]
Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        30
           1       1.00      1.00      1.00        15

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45



**Задание 1.2. Интерпретация (бинарный случай)**

1. Сколько объектов каждого класса модель предсказала правильно / неправильно?  
2. Посмотрите коэффициенты модели и объясните, как изменение каждого признака влияет на вероятность класса setosa.

In [6]:
print("Коэффициенты:", log_reg_iris_bin.coef_)
print("Свободный член:", log_reg_iris_bin.intercept_)
print("Порядок признаков:", ["sepal length (cm)", "petal length (cm)"])

Коэффициенты: [[ 1.31366373 -2.77550204]]
Свободный член: [0.4977141]
Порядок признаков: ['sepal length (cm)', 'petal length (cm)']


### 1.2 Многоклассовая классификация: все три вида ирисов

Теперь будем предсказывать сразу три вида: setosa, versicolor, virginica.

In [7]:
# Используем все 4 признака и исходные метки классов (0, 1, 2)
X_multi = X_iris
y_multi = y_iris

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.3, random_state=42, stratify=y_multi
)

X_train_m.shape, X_test_m.shape

((105, 4), (45, 4))

In [10]:
# Многоклассовая логистическая регрессия
log_reg_iris_multi = LogisticRegression(
    solver="lbfgs",
    max_iter=1000
)

log_reg_iris_multi.fit(X_train_m, y_train_m)

y_pred_m = log_reg_iris_multi.predict(X_test_m)

print("Accuracy (3 класса):", accuracy_score(y_test_m, y_pred_m))
print("Confusion matrix:\n", confusion_matrix(y_test_m, y_pred_m))
print("Classification report:\n",
      classification_report(y_test_m, y_pred_m, target_names=target_names_iris))

Accuracy (3 класса): 0.9333333333333333
Confusion matrix:
 [[15  0  0]
 [ 0 14  1]
 [ 0  2 13]]
Classification report:
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.88      0.93      0.90        15
   virginica       0.93      0.87      0.90        15

    accuracy                           0.93        45
   macro avg       0.93      0.93      0.93        45
weighted avg       0.93      0.93      0.93        45



**Задание 1.3. Многоклассовый случай**

1. Сравните качество модели на трёх классах с бинарным случаем "setosa vs не‑setosa".  
2. Какие виды путаются чаще всего? Почему это может происходить?  
3. Сравните `precision` и `recall` для каждого вида.

In [11]:
print("Форма массива коэффициентов (n_classes, n_features):", log_reg_iris_multi.coef_.shape)
for cls, name in enumerate(target_names_iris):
    print(f"\nКласс {cls} ({name}):")
    print("  coef:", log_reg_iris_multi.coef_[cls])
    print("  intercept:", log_reg_iris_multi.intercept_[cls])

print("\nПорядок признаков:", feature_names_iris)

Форма массива коэффициентов (n_classes, n_features): (3, 4)

Класс 0 (setosa):
  coef: [-0.5450839   0.76445141 -2.22894111 -0.97457755]
  intercept: 9.927747940719014

Класс 1 (versicolor):
  coef: [ 0.42116816 -0.42711738 -0.10046544 -0.83878081]
  intercept: 2.4228793551780066

Класс 2 (virginica):
  coef: [ 0.12391574 -0.33733403  2.32940654  1.81335837]
  intercept: -12.350627295896997

Порядок признаков: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


## 2. Логистическая регрессия для кредитного скоринга

Теперь решим прикладную задачу: прогноз дефолта по кредиту.

In [12]:
credit.head()

,age,income,years_employed,past_due,credit_amount,collateral,default,p_default_true
0,43.047171,58.247620,1.270383,2,307.459869,0,0,0.091568
1,29.600159,41.666652,11.600159,1,195.613639,1,0,0.020575
2,47.504512,48.769642,16.409634,1,301.677391,0,0,0.032650
3,49.405647,82.376463,4.402297,2,251.835197,1,0,0.036799
4,21.000000,60.089866,3.000000,1,368.843862,1,0,0.028572


In [13]:
credit["default"].value_counts(), credit["default"].value_counts(normalize=True)

(default
 0    963
 1     37
 Name: count, dtype: int64,
 default
 0    0.963
 1    0.037
 Name: proportion, dtype: float64)

**Задание 2.1.**  
1. Какова доля дефолтных клиентов?  
2. Является ли выборка сбалансированной по классам?

### 2.1 Подготовка признаков и разбиение на train/test

In [14]:
feature_cols_credit = ["age", "income", "years_employed", "past_due", "credit_amount", "collateral"]
X_c = credit[feature_cols_credit].values
y_c = credit["default"].values

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.3, random_state=42, stratify=y_c
)

X_train_c.shape, X_test_c.shape

((700, 6), (300, 6))

### 2.2 Обучение и оценка качества

In [15]:
log_reg_credit = LogisticRegression(solver="liblinear")

log_reg_credit.fit(X_train_c, y_train_c)

y_pred_c = log_reg_credit.predict(X_test_c)

print("Accuracy:", accuracy_score(y_test_c, y_pred_c))
print("Confusion matrix:\n", confusion_matrix(y_test_c, y_pred_c))
print("Classification report:\n", classification_report(y_test_c, y_pred_c))

Accuracy: 0.9566666666666667
Confusion matrix:
 [[287   2]
 [ 11   0]]
Classification report:
               precision    recall  f1-score   support

           0       0.96      0.99      0.98       289
           1       0.00      0.00      0.00        11

    accuracy                           0.96       300
   macro avg       0.48      0.50      0.49       300
weighted avg       0.93      0.96      0.94       300



**Задание 2.2. Интерпретация скоринговой модели**

1. Какую ошибку для банка вы считаете более опасной:  
   - false positive (отказали хорошему клиенту);  
   - false negative (одобрили плохого клиента)?  
2. Посмотрите коэффициенты модели и определите, какие факторы сильнее всего влияют на вероятность дефолта.

In [16]:
print("Коэффициенты:", log_reg_credit.coef_)
print("Свободный член:", log_reg_credit.intercept_)
print("Порядок признаков:", feature_cols_credit)

Коэффициенты: [[-5.41738802e-02 -1.08684584e-02 -4.57143772e-02  7.14934742e-01
   7.77641360e-04 -1.29485514e+00]]
Свободный член: [-0.84584193]
Порядок признаков: ['age', 'income', 'years_employed', 'past_due', 'credit_amount', 'collateral']


### 2.3 Вероятности и порог принятия решения

In [17]:
y_proba_c = log_reg_credit.predict_proba(X_test_c)[:, 1]

list(zip(y_proba_c[:10], y_pred_c[:10], y_test_c[:10]))

[(np.float64(0.13342825666341887), np.int64(0), np.int64(0)),
 (np.float64(0.055011322853006264), np.int64(0), np.int64(0)),
 (np.float64(0.03391498401304971), np.int64(0), np.int64(0)),
 (np.float64(0.022102937550722995), np.int64(0), np.int64(0)),
 (np.float64(0.04578514020159847), np.int64(0), np.int64(0)),
 (np.float64(0.01659090925506581), np.int64(0), np.int64(1)),
 (np.float64(0.006315345408908834), np.int64(0), np.int64(0)),
 (np.float64(0.02572632193233881), np.int64(0), np.int64(0)),
 (np.float64(0.008748319011778707), np.int64(0), np.int64(0)),
 (np.float64(0.028837311512881898), np.int64(0), np.int64(0))]

По умолчанию используется порог 0.5:  
если вероятность дефолта > 0.5, считаем, что клиент дефолтный.

**Задание 2.3.**  
Измените порог на 0.3 и 0.7 и посмотрите, как меняется матрица ошибок.

In [18]:
def predict_with_threshold(proba, threshold):
    return (proba >= threshold).astype(int)

for thr in [0.3, 0.5, 0.7]:
    y_thr = predict_with_threshold(y_proba_c, thr)
    print(f"\nПорог {thr}")
    print("Confusion matrix:\n", confusion_matrix(y_test_c, y_thr))


Порог 0.3
Confusion matrix:
 [[284   5]
 [ 10   1]]

Порог 0.5
Confusion matrix:
 [[287   2]
 [ 11   0]]

Порог 0.7
Confusion matrix:
 [[289   0]
 [ 11   0]]


**Задание 2.4.**  
1. При каком пороге модель находит больше всего дефолтных клиентов (высокий recall по классу 1)?  
2. При каком пороге меньше всего ошибочно "обиженных" хороших клиентов (ложных отказов)?

## 3. Логистическая регрессия для диагностики болезни растений

Теперь рассмотрим задачу предсказания болезни растений.

In [19]:
plants.head()

,temp,humidity,rain,density,variety,disease,p_disease_true
0,17.054393,75.575005,106.718507,9.878049,2,0,0.006396
1,20.161067,81.502189,222.429218,7.006361,1,0,0.008230
2,28.439626,74.449373,41.551512,3.727385,1,0,0.003512
3,22.969872,50.533393,49.501932,4.143898,1,0,0.003544
4,26.601154,92.275118,69.095033,4.235434,2,0,0.002028


In [20]:
plants["disease"].value_counts(), plants["disease"].value_counts(normalize=True)

(disease
 0    993
 1      7
 Name: count, dtype: int64,
 disease
 0    0.993
 1    0.007
 Name: proportion, dtype: float64)

**Задание 3.1.**  
1. Какова доля больных растений?  
2. Почему важно не пропустить больные растения, даже если появляются ложные тревоги?

### 3.1 Подготовка признаков и разбиение

In [21]:
feature_cols_plants = ["temp", "humidity", "rain", "density", "variety"]
X_p = plants[feature_cols_plants].values
y_p = plants["disease"].values

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_p, y_p, test_size=0.3, random_state=42, stratify=y_p
)

X_train_p.shape, X_test_p.shape

((700, 5), (300, 5))

### 3.2 Обучение и оценка качества

In [22]:
log_reg_plants = LogisticRegression(solver="liblinear")

log_reg_plants.fit(X_train_p, y_train_p)

y_pred_p = log_reg_plants.predict(X_test_p)

print("Accuracy:", accuracy_score(y_test_p, y_pred_p))
print("Confusion matrix:\n", confusion_matrix(y_test_p, y_pred_p))
print("Classification report:\n", classification_report(y_test_p, y_pred_p))

Accuracy: 0.9933333333333333
Confusion matrix:
 [[298   0]
 [  2   0]]
Classification report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00       298
           1       0.00      0.00      0.00         2

    accuracy                           0.99       300
   macro avg       0.50      0.50      0.50       300
weighted avg       0.99      0.99      0.99       300



/home/kalitvin/projects/01-tim/academy-tim-courses/Основы ИИ в АПК/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/kalitvin/projects/01-tim/academy-tim-courses/Основы ИИ в АПК/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/kalitvin/projects/01-tim/academy-tim-courses/Основы ИИ в АПК/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre

**Задание 3.2. Интерпретация**

1. Какое значение имеет метрика `recall` для класса "болен"?  
2. Посмотрите коэффициенты модели и объясните влияние каждого фактора (температура, влажность, осадки, плотность, сорт) на риск болезни.

In [23]:
print("Коэффициенты:", log_reg_plants.coef_)
print("Свободный член:", log_reg_plants.intercept_)
print("Порядок признаков:", feature_cols_plants)

Коэффициенты: [[-0.11748154 -0.04524301 -0.00145461  0.13805575 -0.27122844]]
Свободный член: [-0.34396553]
Порядок признаков: ['temp', 'humidity', 'rain', 'density', 'variety']


### 3.3 Порог для обнаружения болезни

In [24]:
y_proba_p = log_reg_plants.predict_proba(X_test_p)[:, 1]

for thr in [0.3, 0.5, 0.7]:
    y_thr = (y_proba_p >= thr).astype(int)
    print(f"\nПорог {thr}")
    print("Confusion matrix:\n", confusion_matrix(y_test_p, y_thr))


Порог 0.3
Confusion matrix:
 [[298   0]
 [  2   0]]

Порог 0.5
Confusion matrix:
 [[298   0]
 [  2   0]]

Порог 0.7
Confusion matrix:
 [[298   0]
 [  2   0]]


**Задание 3.3.**

1. При каком пороге почти все больные растения обнаруживаются (recall по классу 1 близок к 1)?  
2. Какой порог вы бы рекомендовали агроному, если пропуск болезни приводит к большим потерям урожая?

## 4. Итоговые вопросы

1. В чём принципиальное отличие линейной и логистической регрессии по типу задач и по выходу модели?  
2. В каких из рассмотренных задач важнее всего:
   - высокая общая точность (accuracy);
   - высокая точность положительного класса (precision);
   - высокая полнота положительного класса (recall)?  
3. Как можно использовать предсказанные **вероятности** (а не только метки классов) в реальных задачах:
   - в кредитном скоринге;
   - в диагностике болезней растений;
   - в других прикладных областях?